In [0]:
%run ../_shared/00_project_config

In [0]:
%run ../_shared/01_spark_utils

In [0]:
from pyspark.sql import functions as F


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Project-wide schema names and supported FBref leagues are loaded from
# 00_project_config.
#
# Reusable reading, cleaning, validation, deduplication, and writing helpers
# are loaded from 01_spark_utils.

SILVER_TABLE = f"{SILVER_SCHEMA}.team_standard_stats"

FBREF_DATASET = "standard_stats"

# One Silver row represents:
#
#   team + season + league
#
# for the FBref standard team-statistics dataset.
BUSINESS_KEY = [
    "team_name",
    "season_code",
    "league_code",
]

REQUIRED_SILVER_COLUMNS = [
    "team_name",
    "team_key",
    "season_code",
    "season_start_year",
    "league_code",
]


# -----------------------------------------------------------------------------
# Build expected Bronze table list
# -----------------------------------------------------------------------------
# FBref Bronze maintains one table per:
#
#   dataset + league
#
# Each Bronze table contains all supported seasons for that league.

BRONZE_TABLES = [
    f"{BRONZE_SCHEMA}.fbref_{FBREF_DATASET}_{league_code}"
    for league_code in FBREF_LEAGUES
]


# -----------------------------------------------------------------------------
# Read and combine Bronze tables
# -----------------------------------------------------------------------------
# read_and_union_tables() is strict by default:
#
#   - every expected league table must exist
#   - schemas must be compatible
#
# A missing league should fail the pipeline rather than silently create an
# incomplete Silver dataset.

bronze_df = read_and_union_tables(
    spark,
    BRONZE_TABLES,
)


# -----------------------------------------------------------------------------
# Validate expected FBref source columns
# -----------------------------------------------------------------------------
# FBref is an external HTML source and its table structure can change.
# Validate every field required by this transformation before selecting it.

REQUIRED_BRONZE_COLUMNS = [
    "Squad",
    "#_Pl",
    "Age",
    "Poss",
    "Playing_Time_MP",
    "Playing_Time_Starts",
    "Playing_Time_Min",
    "Playing_Time_90s",
    "Performance_Gls",
    "Performance_Ast",
    "Performance_G+A",
    "Performance_G-PK",
    "Performance_PK",
    "Performance_PKatt",
    "Performance_CrdY",
    "Performance_CrdR",
    "Per_90_Minutes_Gls",
    "Per_90_Minutes_Ast",
    "Per_90_Minutes_G+A",
    "Per_90_Minutes_G-PK",
    "Per_90_Minutes_G+A-PK",
    "_season",
    "_league",
    "_stat",
    "_ingested_at",
    "_source_file",
    "_source_system",
]

validate_columns_exist(
    bronze_df,
    REQUIRED_BRONZE_COLUMNS,
)


# -----------------------------------------------------------------------------
# Clean top-level strings
# -----------------------------------------------------------------------------
# FBref Bronze is already flat, so the shared top-level string cleaner is
# appropriate here.
#
# It trims string values and converts empty strings to NULL.

bronze_df = clean_top_level_strings(bronze_df)


# -----------------------------------------------------------------------------
# Validate dataset identity
# -----------------------------------------------------------------------------
# All rows processed by this notebook should originate from the FBref
# "standard_stats" dataset.

unexpected_stat_rows = (
    bronze_df
    .filter(
        F.col("_stat").isNull()
        | (F.col("_stat") != F.lit(FBREF_DATASET))
    )
    .limit(1)
    .count()
)

if unexpected_stat_rows > 0:
    raise ValueError(
        f"Unexpected _stat value found while processing {FBREF_DATASET}."
    )

# -----------------------------------------------------------------------------
# Validate FBref league identifiers
# -----------------------------------------------------------------------------
# Bronze retains source-specific league codes (pl, ll, bl, sa, lo).
# Before converting them into canonical platform identifiers, verify
# that every source code is supported by the project configuration.
#
# Fail explicitly instead of allowing unknown codes to become NULL
# during the mapping transformation.

invalid_league_rows = (
    bronze_df
    .filter(
        F.col("_league").isNull()
        | ~F.col("_league").isin(*FBREF_LEAGUES.keys())
    )
    .limit(1)
    .count()
)

if invalid_league_rows > 0:
    raise ValueError(
        f"Unsupported FBref league identifier found in {FBREF_DATASET}. "
        f"Expected one of: {sorted(FBREF_LEAGUES.keys())}."
    )

print("FBref source league identifiers validated.")

# -----------------------------------------------------------------------------
# Select, rename, and cast Silver columns
# -----------------------------------------------------------------------------
# Source-oriented FBref names are converted into clearer analytical names.
#
# Compared with the previous version, useful context fields such as possession
# and playing time are retained because they improve downstream analysis and
# make the per-90 statistics easier to interpret.

silver_df = (
    bronze_df
    .select(

        # ---------------------------------------------------------------------
        # Team
        # ---------------------------------------------------------------------

        clean_string("Squad")
        .alias("team_name"),

        F.col("`#_Pl`")
        .cast("int")
        .alias("players_used_count"),

        F.col("Age")
        .cast("double")
        .alias("average_age"),

        F.col("Poss")
        .cast("double")
        .alias("possession_percentage"),


        # ---------------------------------------------------------------------
        # Playing time
        # ---------------------------------------------------------------------

        F.col("Playing_Time_MP")
        .cast("int")
        .alias("matches_played"),

        F.col("Playing_Time_Starts")
        .cast("int")
        .alias("starts"),

        F.col("Playing_Time_Min")
        .cast("int")
        .alias("minutes_played"),

        F.col("Playing_Time_90s")
        .cast("double")
        .alias("nineties_played"),


        # ---------------------------------------------------------------------
        # Performance
        # ---------------------------------------------------------------------

        F.col("Performance_Gls")
        .cast("int")
        .alias("goals_scored"),

        F.col("Performance_Ast")
        .cast("int")
        .alias("assists"),

        F.col("`Performance_G+A`")
        .cast("int")
        .alias("goals_plus_assists"),

        F.col("`Performance_G-PK`")
        .cast("int")
        .alias("non_penalty_goals"),

        F.col("Performance_PK")
        .cast("int")
        .alias("penalty_goals"),

        F.col("Performance_PKatt")
        .cast("int")
        .alias("penalty_kicks_attempted"),

        F.col("Performance_CrdY")
        .cast("int")
        .alias("yellow_cards"),

        F.col("Performance_CrdR")
        .cast("int")
        .alias("red_cards"),


        # ---------------------------------------------------------------------
        # Per-90 performance
        # ---------------------------------------------------------------------

        F.col("Per_90_Minutes_Gls")
        .cast("double")
        .alias("goals_per_90"),

        F.col("Per_90_Minutes_Ast")
        .cast("double")
        .alias("assists_per_90"),

        F.col("`Per_90_Minutes_G+A`")
        .cast("double")
        .alias("goals_plus_assists_per_90"),

        F.col("`Per_90_Minutes_G-PK`")
        .cast("double")
        .alias("non_penalty_goals_per_90"),

        F.col("`Per_90_Minutes_G+A-PK`")
        .cast("double")
        .alias("non_penalty_goals_plus_assists_per_90"),


        # ---------------------------------------------------------------------
        # Season / league context
        # ---------------------------------------------------------------------

        F.col("_season")
        .cast("string")
        .alias("season_code"),

        season_code_to_start_year("_season")
        .alias("season_start_year"),

        # Normalize FBref source league identifiers to canonical values.
        # The mapping is defined centrally in 00_project_config.
        map_values(
            "_league",
            FBREF_LEAGUES,
        ).alias("league_code"),


        # ---------------------------------------------------------------------
        # Bronze lineage metadata
        # ---------------------------------------------------------------------

        F.col("_ingested_at")
        .cast("timestamp")
        .alias("_bronze_ingested_at"),

        F.col("_source_file")
        .cast("string")
        .alias("_bronze_source_file"),

        F.col("_source_system")
        .cast("string")
        .alias("_source_system"),
    )
)


# -----------------------------------------------------------------------------
# Validate business key before deduplication
# -----------------------------------------------------------------------------
# Validate the complete business key before deduplication so rows with NULL
# team, season, or league identifiers cannot be grouped together and reduced
# arbitrarily.

validate_required_not_null(
    silver_df,
    BUSINESS_KEY,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Deduplicate repeated Bronze snapshots
# -----------------------------------------------------------------------------
# Bronze currently uses append mode, so rerunning the same FBref Bronze load
# can produce multiple versions of the same team/season/league record.
#
# Keep the record from the latest Bronze ingestion event.

silver_df = deduplicate_latest(
    silver_df,
    key_columns=BUSINESS_KEY,
    order_column="_bronze_ingested_at",
    tie_breaker_columns=[
        "_bronze_source_file",
    ],
)


# -----------------------------------------------------------------------------
# Resolve canonical team key
# -----------------------------------------------------------------------------

team_mappings_df = load_team_mappings_by_name(
    spark,
    SILVER_SCHEMA,
    "fbref",
)

silver_df = resolve_team_keys_by_name(
    silver_df,
    team_mappings_df,
    source_name_column="team_name",
    output_key_column="team_key",
    fail_on_unmapped=True,
)

print("FBref canonical team resolution completed.")

# -----------------------------------------------------------------------------
# Add Silver processing metadata
# -----------------------------------------------------------------------------

silver_df = silver_df.withColumn(
    "_silver_processed_at",
    F.current_timestamp(),
)


# -----------------------------------------------------------------------------
# Final Silver validation
# -----------------------------------------------------------------------------
# Required analytical identifiers are validated as the final data-quality
# gate.
#
# The helper also returns the final row count, avoiding a separate count()
# action.

validation_metrics = validate_required_not_null(
    silver_df,
    REQUIRED_SILVER_COLUMNS,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Write Silver table
# -----------------------------------------------------------------------------
# The current project phase uses full-refresh Silver processing.

write_full_refresh(
    silver_df,
    SILVER_TABLE,
)


# -----------------------------------------------------------------------------
# Completion message
# -----------------------------------------------------------------------------

print(
    f"Successfully loaded {validation_metrics['row_count']} "
    f"team standard-stat records into {SILVER_TABLE}."
)